<a href="https://colab.research.google.com/github/dilekmirac/dip/blob/main/hafta3_g%C3%B6rev3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time

img = cv2.imread('dusuk_kontrast.jpg', cv2.IMREAD_GRAYSCALE)

if img is None:
    print("Resim bulunamadı. Test için 100-140 değerleri arası sentetik resim üretiliyor...")
    img = np.random.randint(100, 140, (400, 400), dtype=np.uint8)

basla_cv2 = time.perf_counter()

clahe_hazir = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe_cv2 = clahe_hazir.apply(img)

bitir_cv2 = time.perf_counter()
print(f"OpenCV Hazır CLAHE Süresi: {bitir_cv2 - basla_cv2:.4f} sn")
basla_manuel = time.perf_counter()

h, w = img.shape
grid_y, grid_x = 8, 8
clip_limit_float = 2.0

tile_h = h // grid_y
tile_w = w // grid_x

limit_gercek = int(clip_limit_float * (tile_h * tile_w) / 256)
if limit_gercek < 1:
    limit_gercek = 1

luts = np.zeros((grid_y, grid_x, 256), dtype=np.float32)

for i in range(grid_y):
    for j in range(grid_x):

        hist = [0] * 256
        for y in range(i * tile_h, (i + 1) * tile_h):
            for x in range(j * tile_w, (j + 1) * tile_w):
                hist[img[y, x]] += 1

        excess = 0
        for k in range(256):
            if hist[k] > limit_gercek:
                excess += (hist[k] - limit_gercek)
                hist[k] = limit_gercek

        distrib = excess // 256
        remainder = excess % 256

        for k in range(256):
            hist[k] += distrib

        if remainder > 0:
            adim = max(1, 256 // remainder)
            for k in range(0, 256, adim):
                if remainder > 0:
                    hist[k] += 1
                    remainder -= 1

        cdf = [0] * 256
        toplam = 0
        for k in range(256):
            toplam += hist[k]
            cdf[k] = toplam

        cdf_min = 0
        for val in cdf:
            if val > 0:
                cdf_min = val
                break

        toplam_piksel = tile_h * tile_w
        for k in range(256):
            pay = cdf[k] - cdf_min
            payda = toplam_piksel - cdf_min
            if payda > 0:
                yeni_deger = (pay / payda) * 255
                luts[i, j, k] = max(0, min(255, yeni_deger))

img_clahe_manuel = np.zeros_like(img)

for y in range(h):
    for x in range(w):
        p = img[y, x]

        ty = (y / tile_h) - 0.5
        tx = (x / tile_w) - 0.5

        ty1, tx1 = int(np.floor(ty)), int(np.floor(tx))
        ty2, tx2 = ty1 + 1, tx1 + 1

        y1 = max(0, min(grid_y - 1, ty1))
        y2 = max(0, min(grid_y - 1, ty2))
        x1 = max(0, min(grid_x - 1, tx1))
        x2 = max(0, min(grid_x - 1, tx2))

        dy = ty - ty1
        dx = tx - tx1

        p11 = luts[y1, x1, p]
        p12 = luts[y1, x2, p]
        p21 = luts[y2, x1, p]
        p22 = luts[y2, x2, p]

        yeni_piksel = (p11 * (1 - dx) * (1 - dy) +
                       p12 * dx * (1 - dy) +
                       p21 * (1 - dx) * dy +
                       p22 * dx * dy)

        img_clahe_manuel[y, x] = int(yeni_piksel)

bitir_manuel = time.perf_counter()
print(f"Manuel CLAHE Algoritma Süresi: {bitir_manuel - basla_manuel:.4f} sn")

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title("Orijinal Düşük Kontrast")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_clahe_cv2, cmap='gray', vmin=0, vmax=255)
plt.title("OpenCV cv2.createCLAHE()")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_clahe_manuel, cmap='gray', vmin=0, vmax=255)
plt.title("Manuel Algoritma (Hazır Fonsiyonsuz)")
plt.axis('off')

plt.tight_layout()
plt.show()

OpenCV Hazır CLAHE Süresi: 0.0026 sn
Manuel CLAHE Algoritma Süresi: 2.4115 sn
